In [3]:
import pandas as pd
from  transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

c:\Users\Mohit Chavan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

## Data pre-processing

In [11]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)  # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = text.strip().lower()
    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

## Tokenize

In [16]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [17]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data['dialogue'], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding='max_length', max_length=512, truncation=True)

    inputs["labels"] = targets['input_ids']  #token ids => add to input as labels
    return inputs

In [18]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
cal_dataset = val_data.apply(tokenize, axis=1).tolist()

In [19]:
train_dataset[0]

{'input_ids': [836, 6961, 10, 3, 13133, 1240, 226, 6, 103, 25, 341, 3884, 22269, 58, 1240, 226, 10, 7102, 3158, 307, 97, 150, 1616, 3, 10, 61, 17945, 6, 3, 23, 103, 6, 125, 31, 7, 95, 58, 836, 6961, 10, 320, 6, 82, 7930, 19, 479, 21, 841, 113, 228, 3884, 376, 6, 78, 3, 23, 816, 81, 25, 5, 1240, 226, 10, 417, 589, 55, 125, 593, 19, 3, 88, 44, 230, 58, 836, 6961, 10, 3, 107, 51, 233, 614, 12, 817, 6, 43, 29, 31, 17, 1380, 3, 117, 61, 3, 23, 317, 3, 88, 131, 2746, 12, 36, 6720, 295, 6, 78, 3, 88, 1077, 523, 128, 12, 43, 9029, 28, 5, 1240, 226, 10, 150, 682, 6, 8, 589, 19, 3, 23, 163, 43, 1911, 1135, 7, 11, 62, 26, 1496, 1135, 7, 326, 233, 836, 6961, 10, 3, 32, 107, 82, 6, 24, 31, 7, 626, 55, 3, 88, 31, 7, 882, 3, 9, 3164, 4024, 6, 68, 3, 88, 243, 62, 26, 1496, 1135, 2993, 1399, 5, 1240, 226, 10, 248, 258, 55, 2049, 21, 48, 55, 3, 10, 61, 836, 6961, 10, 4244, 1024, 1024, 6, 150, 682, 55, 78, 6, 54, 3, 23, 1499, 376, 39, 381, 58, 1240, 226, 10, 13, 503, 3, 10, 61, 836, 6961, 10, 3, 88, 31, 

In [ ]:
# input ids => dialogue -> token ids

# 1 => EOS, 0 => padding

# attention mask
# labels - target => summary token

In [21]:
len(train_dataset[0]["input_ids"])

512

In [22]:
type(train_dataset)

list